# LeetCode 1324 — Print Words Vertically bằng Python

Link đề: https://leetcode.com/problems/print-words-vertically/ · Độ khó: **Medium** · Chủ đề: Array, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def printVertically(self, s: str) -> List[str]:
        ...
```

## 1. Mục tiêu học

1. Nhìn bài toán như **chuyển vị (transpose) một ma trận ký tự** có các hàng dài ngắn khác nhau.
2. Thành thạo các công cụ chuỗi của Python: `split`, `ljust`, `rstrip`, `"".join`, và hàm `itertools.zip_longest`.
3. Viết được 4 cách: vòng lặp lồng nhau (baseline) → ma trận ký tự đệm khoảng trắng → `zip_longest` một dòng → cách "không cần `rstrip`" bằng cách biết trước cột cuối có chữ.
4. Biết xử lý chi tiết "**không được có khoảng trắng ở cuối**" nhưng **phải giữ khoảng trắng ở đầu/giữa**.
5. Tránh lỗi: quên `rstrip`, dùng nhầm `strip`, dùng `zip` thay `zip_longest`, tạo ma trận bằng `[[' '] * n] * m` (các hàng dùng chung một list).

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các **từ** viết hoa, mỗi hai từ cách nhau đúng **một** dấu cách. Hãy "in dọc" các từ: từ thứ `j` nằm trên **cột** `j`, đọc từ trên xuống. Trả về danh sách các **hàng** (mỗi hàng là một chuỗi). Chỗ nào từ ngắn hơn thì điền dấu cách, nhưng **không được để dấu cách ở cuối hàng**.

**Ví dụ 1:** `s = "HOW ARE YOU"` → `["HAY","ORO","WEU"]`

**Ví dụ 2:** `s = "TO BE OR NOT TO BE"` → `["TBONTB","OEROOE","   T"]` (hàng 3 chỉ có chữ `T` của `NOT`; ba dấu cách đầu phải giữ, phần sau bị cắt).

**Ví dụ 3:** `s = "CONTEST IS COMING"` → `["CIC","OSO","N M","T I","E N","S G","T"]`

**Ràng buộc:** `1 <= s.length <= 200`; `s` chỉ gồm chữ hoa tiếng Anh và dấu cách; giữa hai từ đúng một dấu cách.

In [1]:
import random
import string
import time
from itertools import zip_longest
from typing import List

examples = [
    ("HOW ARE YOU", ["HAY", "ORO", "WEU"]),
    ("TO BE OR NOT TO BE", ["TBONTB", "OEROOE", "   T"]),
    ("CONTEST IS COMING", ["CIC", "OSO", "N M", "T I", "E N", "S G", "T"]),
]
for s, want in examples:
    print(repr(s), "->", want)

'HOW ARE YOU' -> ['HAY', 'ORO', 'WEU']
'TO BE OR NOT TO BE' -> ['TBONTB', 'OEROOE', '   T']
'CONTEST IS COMING' -> ['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']


## 3. Kiến thức nền

### 3.1 `str.split()` — tách từ

`s.split()` (không đối số) tách theo mọi khoảng trắng liên tiếp và bỏ chuỗi rỗng. `s.split(" ")` tách theo **đúng một** dấu cách — nếu có hai dấu cách liền nhau sẽ sinh chuỗi rỗng. Đề đảm bảo đúng một dấu cách nên cả hai đều dùng được, nhưng `split()` an toàn hơn.

In [2]:
print("TO BE OR".split())
print("TO  BE".split(" "), "<- có chuỗi rỗng")
print("TO  BE".split(), "<- an toàn")

['TO', 'BE', 'OR']
['TO', '', 'BE'] <- có chuỗi rỗng
['TO', 'BE'] <- an toàn


### 3.2 Độ dài từ dài nhất: `max(map(len, words))`

Số **hàng** của đáp án = độ dài từ dài nhất. Số **cột** = số từ.

In [3]:
words = "CONTEST IS COMING".split()
print(words, "| độ dài:", list(map(len, words)), "| max =", max(map(len, words)))
print("từ dài nhất:", max(words, key=len))

['CONTEST', 'IS', 'COMING'] | độ dài: [7, 2, 6] | max = 7
từ dài nhất: CONTEST


### 3.3 `ljust`, `rstrip`, `strip`

- `w.ljust(k)` đệm thêm dấu cách **bên phải** cho đủ `k` ký tự.
- `t.rstrip()` xoá khoảng trắng **ở cuối** (bên phải).
- `t.strip()` xoá **cả hai đầu** — **không** dùng cho bài này.

In [4]:
print(repr("IS".ljust(7)))
row = "   T  "
print(repr(row.rstrip()), "<- giữ dấu cách đầu")
print(repr(row.strip()), "<- mất dấu cách đầu (sai cho bài này)")

'IS     '
'   T' <- giữ dấu cách đầu
'T' <- mất dấu cách đầu (sai cho bài này)


### 3.4 `"".join(...)` — ghép ký tự thành chuỗi

Ghép nhiều ký tự bằng `"".join(list_ky_tu)` nhanh hơn cộng chuỗi `+=` trong vòng lặp (mỗi lần `+=` có thể tạo chuỗi mới).

In [5]:
chars = ["N", " ", "M"]
print(repr("".join(chars)))
print(repr("-".join(["A", "B", "C"])))

'N M'
'A-B-C'


### 3.5 `zip` và `itertools.zip_longest`

`zip(*words)` ghép ký tự thứ `i` của mọi từ — nhưng **dừng ở từ ngắn nhất**. `zip_longest(*words, fillvalue=" ")` chạy tới từ **dài nhất**, chỗ thiếu điền `fillvalue`. Đây chính là phép chuyển vị ma trận "răng cưa".

In [6]:
words = ["TO", "BE", "NOT"]
print("zip        :", list(zip(*words)))
print("zip_longest:", list(zip_longest(*words, fillvalue=" ")))

zip        : [('T', 'B', 'N'), ('O', 'E', 'O')]
zip_longest: [('T', 'B', 'N'), ('O', 'E', 'O'), (' ', ' ', 'T')]


### 3.6 Bẫy tạo ma trận 2D

`[[" "] * cols] * rows` tạo `rows` **tham chiếu tới cùng một list** — sửa một hàng là sửa tất cả. Đúng phải là `[[" "] * cols for _ in range(rows)]`.

In [7]:
bad = [[" "] * 3] * 2
bad[0][0] = "X"
good = [[" "] * 3 for _ in range(2)]
good[0][0] = "X"
print("bad :", bad)
print("good:", good)

bad : [['X', ' ', ' '], ['X', ' ', ' ']]
good: [['X', ' ', ' '], [' ', ' ', ' ']]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2**: `"TO BE OR NOT TO BE"` → từ: `TO, BE, OR, NOT, TO, BE` (6 cột), từ dài nhất `NOT` dài 3 ⇒ 3 hàng.

Viết các từ thành **cột**, đệm `·` (dấu cách) cho từ ngắn:

```
cột:   0 1 2 3 4 5
hàng0: T B O N T B   -> "TBONTB"
hàng1: O E R O O E   -> "OEROOE"
hàng2: · · · T · ·   -> "   T  " -> bỏ dấu cách cuối -> "   T"
```

Quan sát: hàng `i` = ghép ký tự thứ `i` của mọi từ (hoặc dấu cách nếu từ quá ngắn), rồi **cắt dấu cách ở cuối**.

In [8]:
s = "TO BE OR NOT TO BE"
words = s.split()
rows = max(map(len, words))
for i in range(rows):
    raw = "".join(w[i] if i < len(w) else " " for w in words)
    print(f"hàng {i}: {raw!r:10} -> {raw.rstrip()!r}")

hàng 0: 'TBONTB'   -> 'TBONTB'
hàng 1: 'OEROOE'   -> 'OEROOE'
hàng 2: '   T  '   -> '   T'


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** làm đúng như tính tay bằng hai vòng lặp lồng nhau: duyệt hàng `i` từ `0` đến `max_len − 1`, với mỗi hàng duyệt từng từ, lấy `w[i]` nếu có, ngược lại lấy `" "`. Cuối hàng gọi `rstrip()`.

In [9]:
def print_vertically_loops(s):
    words = s.split()
    max_len = 0
    for w in words:
        if len(w) > max_len:
            max_len = len(w)
    res = []
    for i in range(max_len):
        row = ""
        for w in words:
            if i < len(w):
                row += w[i]
            else:
                row += " "
        res.append(row.rstrip())
    return res

for s, want in examples:
    got = print_vertically_loops(s)
    print(got, "| OK" if got == want else "| SAI")

['HAY', 'ORO', 'WEU'] | OK
['TBONTB', 'OEROOE', '   T'] | OK
['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T'] | OK


**Độ phức tạp:** gọi `W` = số từ, `L` = độ dài từ dài nhất. Thời gian `O(W·L)` (cộng chi phí `row += ...` — CPython thường tối ưu tại chỗ, nhưng không được đảm bảo). Bộ nhớ `O(W·L)` cho kết quả. Vì `|s| ≤ 200` nên `W·L ≤ 100·200`, rất nhỏ.

**Nút thắt:** ở bài này baseline đã đủ nhanh; điểm yếu là code dài, cộng chuỗi lặp lại, và dễ quên `rstrip`. Các cách sau giúp code **gọn hơn và an toàn hơn**, đồng thời giới thiệu những công cụ dùng lại được cho nhiều bài "chuyển vị".

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đáp án chính là **ma trận chuyển vị** của danh sách từ (coi mỗi từ là một hàng), với các hàng ngắn được đệm dấu cách.

- Nếu **đệm trước** mọi từ tới cùng độ dài `L` (`ljust`), ta có ma trận chữ nhật `W × L` ⇒ chuyển vị bằng `zip(*padded)`.
- Hoặc dùng thẳng `zip_longest(*words, fillvalue=" ")` — vừa đệm vừa chuyển vị.
- Bước cuối luôn là `rstrip()` từng hàng.

**Invariant:** ký tự ở hàng `i`, cột `j` của đáp án (trước khi `rstrip`) = `words[j][i]` nếu `i < len(words[j])`, ngược lại là `" "`.

**Quy tắc "nếu… thì…":**
- Nếu từ `j` dài hơn `i` thì đặt chữ, ngược lại đặt dấu cách.
- Nếu cuối hàng là dấu cách thì cắt; dấu cách **trước** một chữ cái thì phải giữ.
- Muốn **không cần cắt**: chỉ ghép tới cột cuối cùng `j*` có `len(words[j*]) > i` (Cách 4).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ma trận ký tự đệm sẵn rồi chuyển vị

**Từng đoạn:**
1. `L = max(map(len, words))`.
2. Đệm: `padded = [w.ljust(L) for w in words]` ⇒ mọi từ dài `L`.
3. Tạo ma trận `grid[i][j] = padded[j][i]` (kích thước `L × W`).
4. Mỗi hàng: `"".join(grid[i]).rstrip()`.

In [10]:
def print_vertically_grid(s):
    words = s.split()
    L = max(map(len, words))
    W = len(words)
    padded = [w.ljust(L) for w in words]
    grid = [[" "] * W for _ in range(L)]
    for j in range(W):
        for i in range(L):
            grid[i][j] = padded[j][i]
    return ["".join(row).rstrip() for row in grid]

for s, want in examples:
    got = print_vertically_grid(s)
    print(got, "| OK" if got == want else "| SAI")

['HAY', 'ORO', 'WEU'] | OK
['TBONTB', 'OEROOE', '   T'] | OK
['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T'] | OK


Gọn hơn: sau khi đệm, `zip(*padded)` **chính là** phép chuyển vị (mọi "hàng" cùng độ dài nên `zip` không cắt mất gì).

In [11]:
def print_vertically_ljust_zip(s):
    words = s.split()
    L = max(map(len, words))
    padded = [w.ljust(L) for w in words]
    return ["".join(col).rstrip() for col in zip(*padded)]

print(print_vertically_ljust_zip("TO BE OR NOT TO BE"))

['TBONTB', 'OEROOE', '   T']


Code nộp LeetCode (Cách 2):

In [12]:
class SolutionGrid:
    def printVertically(self, s: str) -> List[str]:
        words = s.split()
        L = max(map(len, words))
        padded = [w.ljust(L) for w in words]
        return ["".join(col).rstrip() for col in zip(*padded)]

print([SolutionGrid().printVertically(s) for s, _ in examples])

[['HAY', 'ORO', 'WEU'], ['TBONTB', 'OEROOE', '   T'], ['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']]


**Độ phức tạp:** `O(W·L)` thời gian và bộ nhớ (ma trận đệm).

### 7.2 Cách 3 — `zip_longest` một dòng

`zip_longest(*words, fillvalue=" ")` sinh lần lượt **bộ ký tự thứ `i`** của mọi từ, điền `" "` khi từ đã hết — đúng hàng `i` trước khi cắt. Không cần tính `L`, không cần `ljust`.

In [13]:
def print_vertically_zip_longest(s):
    return ["".join(t).rstrip() for t in zip_longest(*s.split(), fillvalue=" ")]

for s, want in examples:
    got = print_vertically_zip_longest(s)
    print(got, "| OK" if got == want else "| SAI")

['HAY', 'ORO', 'WEU'] | OK
['TBONTB', 'OEROOE', '   T'] | OK
['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T'] | OK


Xem từng bộ mà `zip_longest` sinh ra trên Ví dụ 3:

In [14]:
for i, t in enumerate(zip_longest(*"CONTEST IS COMING".split(), fillvalue=" ")):
    print(i, t, "->", repr("".join(t).rstrip()))

0 ('C', 'I', 'C') -> 'CIC'
1 ('O', 'S', 'O') -> 'OSO'
2 ('N', ' ', 'M') -> 'N M'
3 ('T', ' ', 'I') -> 'T I'
4 ('E', ' ', 'N') -> 'E N'
5 ('S', ' ', 'G') -> 'S G'
6 ('T', ' ', ' ') -> 'T'


Code nộp LeetCode (Cách 3):

In [15]:
class SolutionZipLongest:
    def printVertically(self, s: str) -> List[str]:
        return ["".join(t).rstrip() for t in zip_longest(*s.split(), fillvalue=" ")]

print([SolutionZipLongest().printVertically(s) for s, _ in examples])

[['HAY', 'ORO', 'WEU'], ['TBONTB', 'OEROOE', '   T'], ['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']]


**Độ phức tạp:** `O(W·L)` thời gian, `O(W·L)` bộ nhớ cho kết quả (các bộ tạm chỉ `O(W)` tại mỗi thời điểm).

### 7.3 Cách 4 — Không cần `rstrip`: biết trước cột cuối cùng có chữ

**Ý tưởng:** ở hàng `i`, cột cuối cùng có chữ là từ cuối cùng `j` với `len(words[j]) > i`. Gọi `last[i]` là chỉ số đó. Ta chỉ ghép từ cột `0` đến `last[i]` ⇒ hàng tự động không có dấu cách thừa ở cuối.

Tính `last` trong một lượt: duyệt từ `j = 0..W−1`, với mỗi từ, mọi hàng `i < len(words[j])` có `last[i] = j` (ghi đè dần, từ sau thắng). Tổng công việc `O(tổng độ dài các từ) = O(|s|)`.

**Từng đoạn:**
1. `last = [-1] * L`.
2. Với mỗi `j`, với mỗi `i < len(words[j])`: `last[i] = j`.
3. Hàng `i` = `"".join(words[j][i] if i < len(words[j]) else " " for j in range(last[i] + 1))`.

In [16]:
def print_vertically_no_strip(s):
    words = s.split()
    L = max(map(len, words))
    last = [-1] * L
    for j, w in enumerate(words):
        for i in range(len(w)):
            last[i] = j
    res = []
    for i in range(L):
        res.append("".join(words[j][i] if i < len(words[j]) else " " for j in range(last[i] + 1)))
    return res

for s, want in examples:
    got = print_vertically_no_strip(s)
    print(got, "| OK" if got == want else "| SAI")

['HAY', 'ORO', 'WEU'] | OK
['TBONTB', 'OEROOE', '   T'] | OK
['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T'] | OK


Code nộp LeetCode (Cách 4):

In [17]:
class SolutionNoStrip:
    def printVertically(self, s: str) -> List[str]:
        words = s.split()
        L = max(len(w) for w in words)
        last = [-1] * L
        for j, w in enumerate(words):
            for i in range(len(w)):
                last[i] = j
        return ["".join(words[j][i] if i < len(words[j]) else " "
                        for j in range(last[i] + 1))
                for i in range(L)]

print([SolutionNoStrip().printVertically(s) for s, _ in examples])

[['HAY', 'ORO', 'WEU'], ['TBONTB', 'OEROOE', '   T'], ['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']]


**Độ phức tạp:** vẫn `O(W·L)` trong trường hợp xấu (một từ dài ở cuối buộc mọi hàng phải ghép đủ `W` cột), nhưng thường ít hơn vì không tạo phần đuôi rồi cắt. Bộ nhớ `O(L)` cho `last` + kết quả. Cách này hữu ích khi ngôn ngữ không có `rstrip` tiện lợi, hoặc khi đề yêu cầu một ký tự đệm khác dấu cách mà vẫn cắt đuôi.

## 8. TRACE từng bước

Trace Cách 1 trên **Ví dụ 3** `"CONTEST IS COMING"` (W = 3, L = 7):

| i | CONTEST[i] | IS[i] | COMING[i] | raw | sau rstrip |
|---|---|---|---|---|---|
| 0 | C | I | C | `"CIC"` | `"CIC"` |
| 1 | O | S | O | `"OSO"` | `"OSO"` |
| 2 | N | ␣ | M | `"N M"` | `"N M"` |
| 3 | T | ␣ | I | `"T I"` | `"T I"` |
| 4 | E | ␣ | N | `"E N"` | `"E N"` |
| 5 | S | ␣ | G | `"S G"` | `"S G"` |
| 6 | T | ␣ | ␣ | `"T  "` | `"T"` |

In [18]:
def trace_vertical(s):
    words = s.split()
    L = max(map(len, words))
    print("words =", words, "| L =", L)
    res = []
    for i in range(L):
        parts = []
        for j, w in enumerate(words):
            ch = w[i] if i < len(w) else " "
            parts.append(ch)
        raw = "".join(parts)
        res.append(raw.rstrip())
        print(f"i={i}: parts={parts} raw={raw!r:8} -> {res[-1]!r}")
    return res

trace_vertical("CONTEST IS COMING")

words = ['CONTEST', 'IS', 'COMING'] | L = 7
i=0: parts=['C', 'I', 'C'] raw='CIC'    -> 'CIC'
i=1: parts=['O', 'S', 'O'] raw='OSO'    -> 'OSO'
i=2: parts=['N', ' ', 'M'] raw='N M'    -> 'N M'
i=3: parts=['T', ' ', 'I'] raw='T I'    -> 'T I'
i=4: parts=['E', ' ', 'N'] raw='E N'    -> 'E N'
i=5: parts=['S', ' ', 'G'] raw='S G'    -> 'S G'
i=6: parts=['T', ' ', ' '] raw='T  '    -> 'T'


['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']

Trace Cách 4 trên **Ví dụ 2** — xem mảng `last` được ghi đè:

In [19]:
def trace_last(s):
    words = s.split()
    L = max(map(len, words))
    last = [-1] * L
    for j, w in enumerate(words):
        for i in range(len(w)):
            last[i] = j
        print(f"sau từ j={j} ({w}): last={last}")
    for i in range(L):
        row = "".join(words[j][i] if i < len(words[j]) else " " for j in range(last[i] + 1))
        print(f"hàng {i}: ghép cột 0..{last[i]} -> {row!r}")

trace_last("TO BE OR NOT TO BE")

sau từ j=0 (TO): last=[0, 0, -1]
sau từ j=1 (BE): last=[1, 1, -1]
sau từ j=2 (OR): last=[2, 2, -1]
sau từ j=3 (NOT): last=[3, 3, 3]
sau từ j=4 (TO): last=[4, 4, 3]
sau từ j=5 (BE): last=[5, 5, 3]
hàng 0: ghép cột 0..5 -> 'TBONTB'
hàng 1: ghép cột 0..5 -> 'OEROOE'
hàng 2: ghép cột 0..3 -> '   T'


## 9. Code nộp LeetCode

Bản chọn nộp: `zip_longest` (ngắn, rõ, đúng). Trên LeetCode, `List` và `zip_longest` đã có sẵn trong môi trường; khi chạy local ta import ở đầu notebook.

In [20]:
class Solution:
    def printVertically(self, s: str) -> List[str]:
        words = s.split()
        return ["".join(chars).rstrip()
                for chars in zip_longest(*words, fillvalue=" ")]

for s, want in examples:
    got = Solution().printVertically(s)
    print(got, "| OK" if got == want else "| SAI")

['HAY', 'ORO', 'WEU'] | OK
['TBONTB', 'OEROOE', '   T'] | OK
['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T'] | OK


## 10. Vì sao thuật toán đúng

- **Đúng số hàng:** `zip_longest` sinh đúng `L = max len` bộ — mỗi bộ ứng với một chỉ số ký tự `i`.
- **Đúng nội dung từng ô:** bộ thứ `i` có phần tử thứ `j` là `words[j][i]` nếu tồn tại, ngược lại là `fillvalue = " "` ⇒ trùng với invariant ở mục 6. Như vậy mỗi từ nằm trọn trên **một cột** và mỗi cột chỉ chứa **một từ** — đúng yêu cầu đề.
- **Cắt đúng phần đuôi:** từ chỉ chứa chữ hoa, nên mọi dấu cách trong hàng đều là dấu cách đệm. `rstrip()` chỉ xoá dấu cách sau chữ cái cuối cùng; dấu cách nằm **trước** một chữ cái được giữ nguyên, nên vị trí cột của mọi chữ không bị dịch.
- **Không có hàng rỗng:** hàng `i < L` luôn có ít nhất một từ dài hơn `i` (từ dài nhất) ⇒ sau `rstrip` vẫn còn ít nhất một chữ.

## 11. Phân tích Time / Space

Đặt `n = |s|`, `W` = số từ, `L` = độ dài từ dài nhất (`W·L` có thể lớn hơn `n`, ví dụ một từ dài và nhiều từ ngắn: `W ≈ n/2`, `L ≈ n/2` ⇒ `W·L ≈ n²/4`).

| Cách | Time | Space (ngoài đầu ra) |
|---|---|---|
| 1. Vòng lặp lồng | `O(W·L)` | `O(W)` cho mỗi hàng tạm |
| 2. Đệm + chuyển vị | `O(W·L)` | `O(W·L)` ma trận đệm |
| 3. `zip_longest` | `O(W·L)` | `O(W)` |
| 4. Không `rstrip` | `O(n + kích thước đầu ra)` | `O(L)` |

Kích thước đầu ra (sau khi cắt) cũng có thể tới `Θ(W·L)` (ví dụ từ dài nhất ở **cuối**), nên `O(W·L)` là tối ưu theo trường hợp xấu. Với `n ≤ 200`, mọi cách chạy tức thì.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
solvers = {
    "loops": print_vertically_loops,
    "grid": print_vertically_grid,
    "ljust_zip": print_vertically_ljust_zip,
    "zip_longest": print_vertically_zip_longest,
    "no_strip": print_vertically_no_strip,
    "submit": lambda s: Solution().printVertically(s),
    "submit_grid": lambda s: SolutionGrid().printVertically(s),
    "submit_nostrip": lambda s: SolutionNoStrip().printVertically(s),
}
cases = examples + [
    ("A", ["A"]),                                   # một ký tự
    ("ABC", ["A", "B", "C"]),                       # một từ -> mỗi chữ một hàng
    ("A B C", ["ABC"]),                             # nhiều từ một chữ
    ("A BC DEF", ["ABD", " CE", "  F"]),            # từ dài dần -> dấu cách đầu
    ("DEF BC A", ["DBA", "EC", "F"]),               # từ ngắn dần -> chỉ cắt đuôi
    ("AB C DE", ["ACD", "B E"]),                    # dấu cách ở giữa phải giữ
]
for s, want in cases:
    for name, f in solvers.items():
        assert f(s) == want, (name, s, f(s), want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 9 case qua cho 8 lời giải


### 12.2 Kiểm tra tính hợp lệ + đối chiếu ngẫu nhiên (seed cố định)

Hàm `is_valid(s, rows)` kiểm tra **độc lập** với mọi lời giải: không có dấu cách cuối hàng, không hàng rỗng, và đọc lại từng cột (đệm dấu cách khi hàng ngắn) phải ra đúng các từ.

In [22]:
def is_valid(s, rows):
    words = s.split()
    if any(r != r.rstrip() or r == "" for r in rows):
        return False
    if len(rows) != max(map(len, words)):
        return False
    for j, w in enumerate(words):
        col = "".join(r[j] if j < len(r) else " " for r in rows)
        if col.rstrip() != w or " " in col.rstrip():
            return False
    return all(len(r) <= len(words) for r in rows)

def random_s(rng):
    words = []
    total = -1
    while True:
        w = "".join(rng.choice(string.ascii_uppercase) for _ in range(rng.randint(1, 12)))
        if total + 1 + len(w) > 200:
            break
        words.append(w)
        total += 1 + len(w)
        if rng.random() < 0.15:
            break
    return " ".join(words)

rng = random.Random(1324)
for _ in range(1000):
    s = random_s(rng)
    assert 1 <= len(s) <= 200
    res = {name: tuple(f(s)) for name, f in solvers.items()}
    assert len(set(res.values())) == 1, s
    assert is_valid(s, list(res["submit"])), s
print("1000 chuỗi ngẫu nhiên: mọi lời giải trùng nhau và hợp lệ")

1000 chuỗi ngẫu nhiên: mọi lời giải trùng nhau và hợp lệ


### 12.3 Trường hợp "xấu nhất" của độ dài đầu ra

In [23]:
s_worst = " ".join(["A"] * 50) + " " + "B" * 99      # 50 từ ngắn + 1 từ dài ở cuối, |s| = 199
t0 = time.time()
out = Solution().printVertically(s_worst)
print("len(s) =", len(s_worst), "| số hàng =", len(out), "| tổng ký tự đầu ra =", sum(map(len, out)))
print("hàng 1:", repr(out[1][:10]), "... độ dài", len(out[1]), f"({time.time() - t0:.4f}s)")
assert is_valid(s_worst, out)

len(s) = 199 | số hàng = 99 | tổng ký tự đầu ra = 5049
hàng 1: '          ' ... độ dài 51 (0.0002s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cắt dấu cách cuối

In [24]:
def wrong_no_rstrip(s):
    return ["".join(t) for t in zip_longest(*s.split(), fillvalue=" ")]

s, want = examples[1]
print("sai :", wrong_no_rstrip(s))
print("đúng:", Solution().printVertically(s))

sai : ['TBONTB', 'OEROOE', '   T  ']
đúng: ['TBONTB', 'OEROOE', '   T']


**Vì sao sai:** hàng cuối thành `"   T  "` (có 2 dấu cách cuối) — đề cấm "trailing spaces". **Sửa:** `"".join(t).rstrip()`.

### Lỗi 2 — Dùng `strip()` thay vì `rstrip()`

In [25]:
def wrong_strip(s):
    return ["".join(t).strip() for t in zip_longest(*s.split(), fillvalue=" ")]

print("sai :", wrong_strip(s))
print("đúng:", Solution().printVertically(s))

sai : ['TBONTB', 'OEROOE', 'T']
đúng: ['TBONTB', 'OEROOE', '   T']


**Vì sao sai:** `strip()` xoá cả dấu cách **đầu hàng** ⇒ `"   T"` thành `"T"`, chữ `T` bị dời sang cột 0 (cột của từ `TO`) — sai vị trí. **Sửa:** chỉ dùng `rstrip()`.

### Lỗi 3 — Dùng `zip` thay `zip_longest`

In [26]:
def wrong_zip(s):
    return ["".join(t).rstrip() for t in zip(*s.split())]

for s, want in examples[1:]:
    print("sai :", wrong_zip(s), "| đúng:", Solution().printVertically(s))

sai : ['TBONTB', 'OEROOE'] | đúng: ['TBONTB', 'OEROOE', '   T']
sai : ['CIC', 'OSO'] | đúng: ['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']


**Vì sao sai:** `zip` dừng ở từ **ngắn nhất** (`IS` dài 2) ⇒ mất mọi hàng từ thứ 3 trở đi. **Sửa:** `zip_longest(..., fillvalue=" ")`, hoặc đệm `ljust` trước rồi mới `zip`.

### Lỗi 4 — Tạo ma trận bằng phép nhân list lồng nhau

In [27]:
def wrong_grid_alias(s):
    words = s.split()
    L = max(map(len, words))
    grid = [[" "] * len(words)] * L          # SAI: L hàng là cùng một list
    for j, w in enumerate(words):
        for i, ch in enumerate(w):
            grid[i][j] = ch
    return ["".join(r).rstrip() for r in grid]

s = "CONTEST IS COMING"
print("sai :", wrong_grid_alias(s))
print("đúng:", Solution().printVertically(s))

sai : ['TSG', 'TSG', 'TSG', 'TSG', 'TSG', 'TSG', 'TSG']
đúng: ['CIC', 'OSO', 'N M', 'T I', 'E N', 'S G', 'T']


**Vì sao sai:** mọi hàng là **cùng một đối tượng list**, ghi vào `grid[i][j]` thực chất ghi vào hàng duy nhất ⇒ mọi hàng giống nhau và chứa chữ ghi sau cùng. **Sửa:** `grid = [[" "] * len(words) for _ in range(L)]`.

### Lỗi 5 — Tính số hàng bằng độ dài từ đầu tiên

In [28]:
def wrong_rows_first_word(s):
    words = s.split()
    L = len(words[0])                     # SAI: phải là max độ dài
    return ["".join(w[i] if i < len(w) else " " for w in words).rstrip() for i in range(L)]

s = "TO BE OR NOT TO BE"
print("sai :", wrong_rows_first_word(s), "| đúng:", Solution().printVertically(s))

sai : ['TBONTB', 'OEROOE'] | đúng: ['TBONTB', 'OEROOE', '   T']


**Vì sao sai:** từ đầu `TO` dài 2 nhưng `NOT` dài 3 ⇒ mất hàng `"   T"`. **Sửa:** `L = max(map(len, words))`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề nói "**in dọc**", "đọc theo cột", "xoay", "đổi hàng thành cột" ⇒ **chuyển vị**: `zip(*rows)`.
- Các hàng **dài ngắn khác nhau** ⇒ `zip_longest(*rows, fillvalue=...)` hoặc `ljust` trước.
- "Không có khoảng trắng cuối" ⇒ `rstrip()`; "không có khoảng trắng đầu" ⇒ `lstrip()`; đừng dùng `strip()` khi chỉ cần một phía.
- Chuỗi ngắn (`≤ 200`) ⇒ ưu tiên code **rõ ràng, ít lỗi** hơn là tối ưu.
- Kiểm thử bài định dạng chuỗi: luôn in bằng `repr()` để **thấy** dấu cách.

## 15. Biến thể của bài

1. **LeetCode 867 — Transpose Matrix:** chuyển vị ma trận số chữ nhật — `[list(r) for r in zip(*matrix)]`.
2. **LeetCode 6 — Zigzag Conversion:** ghi chuỗi theo đường zigzag rồi đọc từng hàng.
3. **In dọc căn đáy** (từ ngắn được đẩy xuống dưới): đệm bên **trái** bằng `rjust`.
4. **Ký tự đệm khác** (ví dụ `"*"`) và vẫn cắt đuôi: dùng `rstrip("*")` hoặc Cách 4.
5. **Phép ngược:** cho các hàng đã in dọc, khôi phục câu gốc (Bài tập 5).

In [29]:
def print_vertically_bottom(s):
    """Biến thể: căn đáy — từ ngắn nằm sát dưới."""
    words = s.split()
    L = max(map(len, words))
    padded = [w.rjust(L) for w in words]
    return ["".join(col).rstrip() for col in zip(*padded)]

def transpose(matrix):
    return [list(r) for r in zip(*matrix)]

for row in print_vertically_bottom("TO BE OR NOT TO BE"):
    print(repr(row))
print(transpose([[1, 2, 3], [4, 5, 6]]))

'   N'
'TBOOTB'
'OERTOE'
[[1, 4], [2, 5], [3, 6]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp lồng | hàng `i` = ký tự `i` của mỗi từ hoặc `" "`, rồi `rstrip` | `O(W·L)` | `O(W)` | mới học, phỏng vấn bằng ngôn ngữ bất kỳ |
| 2. Đệm `ljust` + chuyển vị | ma trận chữ nhật → `zip(*padded)` | `O(W·L)` | `O(W·L)` | khi cần cả ma trận để xử lý thêm |
| 3. `zip_longest` | đệm + chuyển vị trong một lệnh | `O(W·L)` | `O(W)` | Python, code ngắn nhất |
| 4. Không `rstrip` | biết cột cuối có chữ `last[i]` | `O(n + output)` | `O(L)` | ký tự đệm đặc biệt / ngôn ngữ không có `rstrip` |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().printVertically("A BC DEF")` trả về gì? (Chú ý dấu cách đầu.)

### Bài 2 (tính tay)
Tính tay `printVertically("I AM HAPPY")`, ghi rõ từng hàng bằng `repr`.

### Bài 3 (điền TODO)
Hoàn thiện `vertical_todo(s)` dùng `ljust` + `zip`.

### Bài 4 (tự viết lại)
Viết lại Cách 4 (không dùng `rstrip`) mà không nhìn code: `vertical_no_strip_ex(s)`.

### Bài 5 (mini challenge)
Viết `unprint(rows)` khôi phục chuỗi `s` gốc từ đầu ra của `printVertically`. Kiểm tra vòng: `unprint(printVertically(s)) == s` với 500 chuỗi ngẫu nhiên.

In [30]:
# Bài 3 & 4 & 5 — khung (chỉ định nghĩa, không gọi)
def vertical_todo(s):
    words = s.split()
    L = None            # TODO: độ dài từ dài nhất
    padded = None       # TODO: đệm mỗi từ tới độ dài L
    # TODO: chuyển vị bằng zip, join, rstrip
    pass

def vertical_no_strip_ex(s):
    # TODO
    pass

def unprint_todo(rows):
    # TODO: số từ = độ dài hàng đầu tiên; đọc từng cột
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) 3 từ, từ dài nhất 3 chữ ⇒ 3 hàng. (2) Hàng 1: từ `A` đã hết ⇒ dấu cách đầu. (3) `["ABD", " CE", "  F"]`.

**Bài 2:** (1) Từ: `I, AM, HAPPY`; `L = 5`. (2) Hàng 0 `"IAH"`, hàng 1 `" MA"`. (3) Ba hàng cuối chỉ có chữ của `HAPPY` ở cột 2 ⇒ `"  P"`, `"  P"`, `"  Y"`.

**Bài 3:** (1) `L = max(map(len, words))`. (2) `padded = [w.ljust(L) for w in words]`. (3) `return ["".join(c).rstrip() for c in zip(*padded)]`.

**Bài 4:** (1) Mảng `last` độ dài `L`. (2) Với từ `j`, mọi `i < len(w)` gán `last[i] = j`. (3) Hàng `i` ghép cột `0..last[i]`.

**Bài 5:** (1) Hàng 0 luôn đủ `W` chữ (mọi từ có ≥ 1 chữ) ⇒ `W = len(rows[0])`. (2) Cột `j`: ghép `r[j]` cho các hàng có `j < len(r)`, bỏ dấu cách. (3) Nối các từ bằng một dấu cách.

## 19. Đáp án tham khảo

In [31]:
# Bài 1, 2
print("Bài 1:", Solution().printVertically("A BC DEF"))
print("Bài 2:", [repr(r) for r in Solution().printVertically("I AM HAPPY")])
assert Solution().printVertically("A BC DEF") == ["ABD", " CE", "  F"]
assert Solution().printVertically("I AM HAPPY") == ["IAH", " MA", "  P", "  P", "  Y"]

# Bài 3
def vertical_sol(s):
    words = s.split()
    L = max(map(len, words))
    padded = [w.ljust(L) for w in words]
    return ["".join(c).rstrip() for c in zip(*padded)]

# Bài 4
def vertical_no_strip_sol(s):
    words = s.split()
    L = max(map(len, words))
    last = [-1] * L
    for j, w in enumerate(words):
        for i in range(len(w)):
            last[i] = j
    return ["".join(words[j][i] if i < len(words[j]) else " " for j in range(last[i] + 1))
            for i in range(L)]

for s, want in cases:
    assert vertical_sol(s) == want and vertical_no_strip_sol(s) == want
print("Bài 1–4: qua")

Bài 1: ['ABD', ' CE', '  F']
Bài 2: ["'IAH'", "' MA'", "'  P'", "'  P'", "'  Y'"]
Bài 1–4: qua


In [32]:
# Bài 5
def unprint(rows):
    W = len(rows[0])
    words = []
    for j in range(W):
        words.append("".join(r[j] for r in rows if j < len(r)).replace(" ", ""))
    return " ".join(words)

print("Bài 5:", unprint(["TBONTB", "OEROOE", "   T"]))
rng = random.Random(5)
for _ in range(500):
    s = random_s(rng)
    assert unprint(Solution().printVertically(s)) == s
print("Bài 5: 500 chuỗi ngẫu nhiên khôi phục đúng")

Bài 5: TO BE OR NOT TO BE


Bài 5: 500 chuỗi ngẫu nhiên khôi phục đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template chuyển vị hàng răng cưa:**

```python
from itertools import zip_longest
rows = ["".join(t).rstrip() for t in zip_longest(*words, fillvalue=" ")]
# hoặc
L = max(map(len, words))
rows = ["".join(c).rstrip() for c in zip(*(w.ljust(L) for w in words))]
```

**Checklist trước khi submit:**
- [ ] Tách từ bằng `split()`.
- [ ] Số hàng = độ dài từ **dài nhất**.
- [ ] Dùng `zip_longest` (hoặc `ljust` trước `zip`), **không** dùng `zip` trần.
- [ ] Cắt đuôi bằng `rstrip()`, **không** dùng `strip()`.
- [ ] Nếu tạo ma trận: dùng list comprehension, không `[[...]*a]*b`.
- [ ] Kiểm thử bằng `repr()` để thấy dấu cách.
- [ ] Đúng chữ ký: `printVertically(self, s: str) -> List[str]`.